In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

print("EDA environment loaded successfully")

Connect SqlLite

In [ ]:
connection = sqlite3.connect("../database/HospitalDB.db")

Load Tables

<!-- Load Patients -->
patients = pd.read_sql_query(
    "SELECT * FROM Patients",
    connection
)

Load all healthcare tables

In [ ]:
patients = pd.read_sql_query(
    "SELECT * FROM Patients",
    connection
)

doctors = pd.read_sql_query(
    "SELECT * FROM Doctors",
    connection
)

departments = pd.read_sql_query(
    "SELECT * FROM Departments",
    connection
)

appointments = pd.read_sql_query(
    "SELECT * FROM Appointments",
    connection
)

Data Inspection

In [ ]:
patients.head()

In [ ]:
patients.tail()

shape

In [ ]:
patients.shape

columns

In [ ]:
patients.columns

data types check

patients.dtypes

patients.info()

statistical details

In [ ]:
patients.describe()

Missing values

missing_values = patients.isnull().sum()

missing_values

<!-- Percentage -->
missing_percentage = (
    patients.isnull().mean() * 100
).sort_values(ascending=False)

missing_percentage

Patient demographics

In [ ]:
patients["Gender"].value_counts()

In [ ]:
patients["Age"].describe()

Age distribution — Matplotlib

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    patients["Age"],
    bins=20,
    edgecolor="black"
)

plt.title("Patient Age Distribution")
plt.xlabel("Age")
plt.ylabel("Number of Patients")

plt.show()

Gender distribution — Seaborn

In [ ]:
plt.figure(figsize=(7,5))

sns.countplot(
    data=patients,
    x="Gender"
)

plt.title("Patient Gender Distribution")
plt.xlabel("Gender")
plt.ylabel("Number of Patients")

plt.show()

Diabetes distribution

In [ ]:
patients["Diabetes"].value_counts()

In [ ]:
plt.figure(figsize=(7,5))

sns.countplot(
    data=patients,
    x="Diabetes"
)

plt.title("Diabetes Distribution")
plt.xlabel("Diabetes")
plt.ylabel("Number of Patients")

plt.show()

BMI distribution

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    data=patients,
    x="BMI",
    kde=True
)

plt.title("BMI Distribution")
plt.xlabel("BMI")
plt.ylabel("Frequency")

plt.show()

Diabetes vs BMI

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=patients,
    x="Diabetes",
    y="BMI"
)

plt.title("BMI Distribution by Diabetes Status")

plt.show()

Glucose vs Diabetes

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=patients,
    x="Diabetes",
    y="Glucose"
)

plt.title("Glucose Levels by Diabetes Status")

plt.show()

Correlation analysis

In [ ]:
numeric_columns = [
    "Age",
    "BMI",
    "BloodPressure",
    "Cholesterol",
    "Glucose"
]

correlation = patients[numeric_columns].corr()

correlation

In [ ]:
plt.figure(figsize=(8,6))

sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Healthcare Feature Correlation")

plt.show()

In [ ]:
healthcare_data = pd.read_sql_query("""
SELECT
    p.PatientID,
    p.PatientName,
    p.Age,
    p.Gender,
    p.BMI,
    p.BloodPressure,
    p.Cholesterol,
    p.Glucose,
    p.Smoking,
    p.Diabetes,
    d.DoctorName,
    dep.DepartmentName,
    a.AppointmentDate,
    a.Diagnosis
FROM Patients p
LEFT JOIN Doctors d
    ON p.DoctorID = d.DoctorID
LEFT JOIN Departments dep
    ON d.DepartmentID = dep.DepartmentID
LEFT JOIN Appointments a
    ON p.PatientID = a.PatientID
""", connection)

healthcare_data.head()

Department analysis

In [ ]:
department_counts = (
    healthcare_data["DepartmentName"]
    .value_counts()
)

department_counts

In [ ]:
plt.figure(figsize=(10,6))

sns.countplot(
    data=healthcare_data,
    y="DepartmentName",
    order=department_counts.index
)

plt.title("Patients by Department")
plt.xlabel("Number of Records")
plt.ylabel("Department")

plt.show()

Now basic Machine Learning

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

Prepare ML dataset

In [ ]:
ml_data = patients[
    [
        "Age",
        "BMI",
        "BloodPressure",
        "Cholesterol",
        "Glucose",
        "Smoking",
        "Diabetes"
    ]
].copy()

ml_data.head()

check column value in numbers or text

In [ ]:
ml_data = patients[
    [
        "Age",
        "BMI",
        "BloodPressure",
        "Cholesterol",
        "Glucose",
        "Smoking",
        "Diabetes"
    ]
].copy()

ml_data.head()

encoding

In [ ]:
ml_data["Smoking"] = ml_data["Smoking"].map({
    "Yes": 1,
    "No": 0
})

ml_data["Diabetes"] = ml_data["Diabetes"].map({
    "Yes": 1,
    "No": 0
})

Train/test split

In [ ]:
X = ml_data.drop("Diabetes", axis=1)
y = ml_data["Diabetes"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

ML Pipeline

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression())
])

model.fit(X_train, y_train)

print("Model trained successfully")

Evaluate

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print(classification_report(y_test, y_pred))

Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Diabetes Prediction Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

close database

In [ ]:
connection.close()

print("Database connection closed")
print("EDA and baseline ML pipeline completed successfully.")